# 06 - Adversarial Training
## CSE-CIC-IDS2018 — Penguatan Model NIDS via Data Augmentation

**Tujuan:** Melatih ulang model XGBoost dengan dataset gabungan (clean + adversarial)
untuk memperkuat decision boundary terhadap serangan evasion.

**Metodologi (dari paper):**
1. Konstruksi D_robust = D_clean ∪ D_adv (rasio 80:20)
2. Retrain XGBoost pada D_robust
3. Evaluasi: S3 (robust + clean) dan S4 (robust + adversarial)
4. Bandingkan dengan baseline (S1, S2 dari Notebook 05)

**Input:**
- `adversarial_samples_05.pkl` — adversarial samples dari Notebook 05
- `cleaned_100.pkl` — dataset bersih
- `experiment_results_03.pkl` — feature lists, label mapping

**Output:**
- `robust_model_06.json` — model XGBoost yang diperkuat
- `robust_results_06.pkl` — hasil evaluasi model robust
- Visualisasi: perbandingan S1-S4

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn xgboost matplotlib seaborn numpy pandas -q
print('✓ Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, json, time, warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    classification_report
)
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

DATA_DIR = '../data/'
MODEL_DIR = '../models/'
RANDOM_SEED = 42

print('Libraries loaded ✓')

## 1. Load Adversarial Samples & Baseline Data

In [ ]:
# Load adversarial output dari Notebook 05
print('Loading adversarial_samples_05.pkl...')
with open(os.path.join(DATA_DIR, 'adversarial_samples_05.pkl'), 'rb') as f:
    adv_data = pickle.load(f)

# Extract data
X_train_clean = adv_data['X_train']
y_train_clean = adv_data['y_train'] if isinstance(adv_data['y_train'], np.ndarray) else adv_data['y_train'].values
X_test_clean = adv_data['X_test_clean']
y_test = adv_data['y_test'] if isinstance(adv_data['y_test'], np.ndarray) else adv_data['y_test'].values

# Adversarial samples for training (ε=0.1 pada training set)
X_adv_train = adv_data['X_adv_eps10_train']
y_adv_train = adv_data['y_adv_train'] if isinstance(adv_data['y_adv_train'], np.ndarray) else adv_data['y_adv_train'].values

# Adversarial test set (untuk evaluasi S4)
X_test_adv = adv_data['X_adv_eps10']  # ε=0.1 pada test set

# Config
top10_features = adv_data['top10_features']
label_mapping = adv_data['label_mapping']
inverse_label = {v: k for k, v in label_mapping.items()}
EPSILON = adv_data['epsilon_for_training']

# Baseline performance (dari NB05)
baseline_mcc = adv_data['baseline_mcc_clean']
baseline_f1 = adv_data['baseline_f1_clean']

print(f'\nData loaded:')
print(f'  X_train_clean: {X_train_clean.shape}')
print(f'  X_adv_train:   {X_adv_train.shape}')
print(f'  X_test_clean:  {X_test_clean.shape}')
print(f'  X_test_adv:    {X_test_adv.shape}')
print(f'  Epsilon:       {EPSILON}')
print(f'  Features:      {len(top10_features)}')
print(f'  Baseline MCC:  {baseline_mcc:.4f}')
print(f'  Baseline F1:   {baseline_f1*100:.2f}%')

## 2. Konstruksi Dataset Robust: D_robust = D_clean ∪ D_adv

Rasio augmentasi: **80% clean + 20% adversarial** (sesuai paper Bab 2.4)

In [ ]:
# Tentukan jumlah adversarial samples (20% dari total)
# Total robust = clean + adv
# adv / (clean + adv) = 0.20 → adv = 0.25 * clean
n_clean = len(X_train_clean)
n_adv_target = int(n_clean * 0.25)  # 20% of total = 25% of clean

# Jika adversarial training samples kurang, gunakan semua yang ada
n_adv_available = len(X_adv_train)
n_adv_use = min(n_adv_target, n_adv_available)

print(f'Augmentation Strategy:')
print(f'  Clean training samples:       {n_clean:,}')
print(f'  Target adversarial (20%):     {n_adv_target:,}')
print(f'  Available adversarial:        {n_adv_available:,}')
print(f'  Using adversarial:            {n_adv_use:,}')
print(f'  Final ratio: {n_clean/(n_clean+n_adv_use)*100:.1f}% clean + {n_adv_use/(n_clean+n_adv_use)*100:.1f}% adv')

# Random sample adversarial jika lebih dari yang dibutuhkan
if n_adv_available > n_adv_target:
    np.random.seed(RANDOM_SEED)
    adv_indices = np.random.choice(n_adv_available, n_adv_use, replace=False)
    X_adv_selected = X_adv_train[adv_indices]
    y_adv_selected = y_adv_train[adv_indices]
else:
    X_adv_selected = X_adv_train[:n_adv_use]
    y_adv_selected = y_adv_train[:n_adv_use]

# Gabungkan: D_robust = D_clean ∪ D_adv
X_robust = np.vstack([X_train_clean, X_adv_selected])
y_robust = np.concatenate([y_train_clean, y_adv_selected])

# Shuffle
shuffle_idx = np.random.RandomState(RANDOM_SEED).permutation(len(X_robust))
X_robust = X_robust[shuffle_idx]
y_robust = y_robust[shuffle_idx]

print(f'\nD_robust constructed:')
print(f'  Shape: {X_robust.shape}')
print(f'  Total samples: {len(X_robust):,}')
print(f'  Class distribution preserved: ✓ (labels from both clean & adv share same y)')

## 3. Training Model Robust (Adversarial Training)

In [ ]:
# Konfigurasi model (sama dengan baseline untuk fair comparison)
n_classes = len(np.unique(y_robust))

model_robust = XGBClassifier(
    n_estimators=200, 
    max_depth=8, 
    learning_rate=0.1,
    subsample=0.8, 
    colsample_bytree=0.8,
    objective='multi:softprob', 
    num_class=n_classes,
    eval_metric='mlogloss', 
    random_state=RANDOM_SEED,
    n_jobs=-1, 
    tree_method='hist'
)

print('Training Robust Model (Adversarial Training)...')
print(f'  Dataset: {X_robust.shape[0]:,} samples ({len(top10_features)} features)')
print(f'  Hyperparameters: n_est=200, depth=8, lr=0.1')

start = time.time()
model_robust.fit(X_robust, y_robust)
train_time = time.time() - start

print(f'  Training time: {train_time:.2f}s')
print('  Training complete ✓')

In [ ]:
# Save robust model
robust_model_path = os.path.join(MODEL_DIR, 'robust_xgboost_top10.json')
model_robust.save_model(robust_model_path)
model_size = os.path.getsize(robust_model_path) / (1024*1024)
print(f'Saved: robust_xgboost_top10.json ({model_size:.2f} MB)')

## 4. Evaluasi Skenario 2×2 (S1-S4)

| Skenario | Model | Data Uji | Tujuan |
|----------|-------|----------|--------|
| S1 | Baseline | Clean | Performa awal |
| S2 | Baseline | Adversarial | Kerentanan (security gap) |
| S3 | Robust | Clean | Integritas (akurasi terjaga?) |
| S4 | Robust | Adversarial | Robustness (pemulihan?) |

In [ ]:
# Load baseline model
DEPLOY_DIR = os.path.join(MODEL_DIR, 'deploy')
deploy_files = os.listdir(DEPLOY_DIR) if os.path.exists(DEPLOY_DIR) else []
xgb_top10_file = [f for f in deploy_files if 'xgboost' in f and 'top-10' in f and f.endswith('.json')]

model_baseline = XGBClassifier()
if xgb_top10_file:
    model_baseline.load_model(os.path.join(DEPLOY_DIR, xgb_top10_file[0]))
    print(f'Loaded baseline: {xgb_top10_file[0]}')
else:
    # Retrain baseline
    model_baseline = XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', random_state=RANDOM_SEED,
        n_jobs=-1, tree_method='hist'
    )
    model_baseline.fit(X_train_clean, y_train_clean)
    print('Retrained baseline model')

In [ ]:
def full_evaluation(model, X, y_true, scenario_name):
    """
    Evaluasi lengkap dengan semua metrik.
    """
    y_pred = model.predict(X)
    
    results = {
        'scenario': scenario_name,
        'mcc': matthews_corrcoef(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
        'y_pred': y_pred,
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    return results


# Run all 4 scenarios
print('='*80)
print(f'{"EVALUASI 2×2: BASELINE vs ROBUST × CLEAN vs ADVERSARIAL":^80}')
print('='*80)

results_2x2 = {}

# S1: Baseline + Clean
results_2x2['S1'] = full_evaluation(model_baseline, X_test_clean, y_test, 'S1: Baseline+Clean')
print(f'\n  S1 (Baseline + Clean):       MCC={results_2x2["S1"]["mcc"]:.4f} | F1={results_2x2["S1"]["f1_score"]*100:.2f}%')

# S2: Baseline + Adversarial
results_2x2['S2'] = full_evaluation(model_baseline, X_test_adv, y_test, 'S2: Baseline+Adv')
print(f'  S2 (Baseline + Adversarial): MCC={results_2x2["S2"]["mcc"]:.4f} | F1={results_2x2["S2"]["f1_score"]*100:.2f}% ← VULNERABILITY')

# S3: Robust + Clean
results_2x2['S3'] = full_evaluation(model_robust, X_test_clean, y_test, 'S3: Robust+Clean')
print(f'  S3 (Robust + Clean):         MCC={results_2x2["S3"]["mcc"]:.4f} | F1={results_2x2["S3"]["f1_score"]*100:.2f}% ← INTEGRITY CHECK')

# S4: Robust + Adversarial
results_2x2['S4'] = full_evaluation(model_robust, X_test_adv, y_test, 'S4: Robust+Adv')
print(f'  S4 (Robust + Adversarial):   MCC={results_2x2["S4"]["mcc"]:.4f} | F1={results_2x2["S4"]["f1_score"]*100:.2f}% ← ROBUSTNESS')

# Summary
security_gap = results_2x2['S1']['mcc'] - results_2x2['S2']['mcc']
recovery = results_2x2['S4']['mcc'] - results_2x2['S2']['mcc']
integrity_loss = results_2x2['S1']['mcc'] - results_2x2['S3']['mcc']

print(f'\n  → Security Gap (S1-S2):    {security_gap:.4f} MCC')
print(f'  → Recovery (S4-S2):        +{recovery:.4f} MCC')
print(f'  → Integrity Loss (S1-S3):  {integrity_loss:.4f} MCC {"✓ minimal" if abs(integrity_loss) < 0.02 else "⚠ significant"}')

In [ ]:
# Tabel lengkap
print('\n'+'='*95)
print(f'{"TABLE: EVALUASI 2×2 — ADVERSARIAL TRAINING EFFECTIVENESS":^95}')
print('='*95)
print(f'{"Skenario":<25s} {"Model":<10s} {"Data":<12s} {"MCC":>7s} {"F1 (%)":>8s} {"Prec (%)":>9s} {"Rec (%)":>8s} {"Acc (%)":>8s}')
print('-'*95)

for key in ['S1', 'S2', 'S3', 'S4']:
    r = results_2x2[key]
    model_type = 'Baseline' if key in ['S1', 'S2'] else 'Robust'
    data_type = 'Clean' if key in ['S1', 'S3'] else 'Adversarial'
    print(f'{r["scenario"]:<25s} {model_type:<10s} {data_type:<12s} '
          f'{r["mcc"]:>7.4f} {r["f1_score"]*100:>7.2f}% '
          f'{r["precision"]*100:>8.2f}% {r["recall"]*100:>7.2f}% '
          f'{r["accuracy"]*100:>7.2f}%')

print('='*95)

## 5. Visualisasi: Perbandingan S1-S4

In [ ]:
# Bar chart: MCC comparison
scenarios = ['S1\n(Base+Clean)', 'S2\n(Base+Adv)', 'S3\n(Robust+Clean)', 'S4\n(Robust+Adv)']
mcc_values = [results_2x2[k]['mcc'] for k in ['S1', 'S2', 'S3', 'S4']]
f1_values = [results_2x2[k]['f1_score']*100 for k in ['S1', 'S2', 'S3', 'S4']]

colors = ['steelblue', 'crimson', 'forestgreen', 'darkorange']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MCC
bars1 = ax1.bar(scenarios, mcc_values, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('MCC', fontsize=11)
ax1.set_title('Matthews Correlation Coefficient\nper Skenario Evaluasi', fontsize=12, fontweight='bold')
ax1.set_ylim([0, 1.1])
ax1.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='Target (0.90)')
ax1.legend()
for bar, val in zip(bars1, mcc_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

# F1
bars2 = ax2.bar(scenarios, f1_values, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('F1-Score (%)', fontsize=11)
ax2.set_title('F1-Score (Weighted)\nper Skenario Evaluasi', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 110])
ax2.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='Target (90%)')
ax2.legend()
for bar, val in zip(bars2, f1_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'adversarial_training_2x2_comparison.png'), bbox_inches='tight')
plt.show()
print('Saved: adversarial_training_2x2_comparison.png')

In [ ]:
# Confusion Matrix: S2 vs S4 (vulnerability vs robustness)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

class_names = [inverse_label.get(int(c), f'C{c}') for c in np.unique(y_test)]

# S2: Baseline + Adversarial (vulnerable)
cm_s2 = results_2x2['S2']['confusion_matrix']
cm_s2_norm = cm_s2.astype('float') / cm_s2.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_s2_norm, annot=True, fmt='.2f', cmap='Reds', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title(f'S2: Baseline + Adversarial\nMCC={results_2x2["S2"]["mcc"]:.4f} (VULNERABLE)', 
                  fontweight='bold', color='red')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# S4: Robust + Adversarial (hardened)
cm_s4 = results_2x2['S4']['confusion_matrix']
cm_s4_norm = cm_s4.astype('float') / cm_s4.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_s4_norm, annot=True, fmt='.2f', cmap='Greens', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title(f'S4: Robust + Adversarial\nMCC={results_2x2["S4"]["mcc"]:.4f} (HARDENED)', 
                  fontweight='bold', color='green')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.suptitle(f'Adversarial Training Effectiveness: Before vs After\n(ε={EPSILON})', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'confusion_s2_vs_s4.png'), bbox_inches='tight')
plt.show()
print('Saved: confusion_s2_vs_s4.png')

## 6. Per-Class Recovery Analysis

In [ ]:
print('\nPer-Class Performance Recovery (S2 → S4):')
print('='*90)
print(f'{"Class":<25s} {"S1 F1":>8s} {"S2 F1":>8s} {"S3 F1":>8s} {"S4 F1":>8s} {"Recovery":>10s} {"Status":>10s}')
print('-'*90)

per_class_recovery = []
for cls in np.unique(y_test):
    cls_name = inverse_label.get(int(cls), f'Class {cls}')
    mask = y_test == cls
    
    y_true_bin = (y_test[mask] == cls).astype(int)
    
    f1_s1 = f1_score(y_true_bin, (results_2x2['S1']['y_pred'][mask] == cls).astype(int), zero_division=0)
    f1_s2 = f1_score(y_true_bin, (results_2x2['S2']['y_pred'][mask] == cls).astype(int), zero_division=0)
    f1_s3 = f1_score(y_true_bin, (results_2x2['S3']['y_pred'][mask] == cls).astype(int), zero_division=0)
    f1_s4 = f1_score(y_true_bin, (results_2x2['S4']['y_pred'][mask] == cls).astype(int), zero_division=0)
    
    recovery = f1_s4 - f1_s2
    status = '✓ RECOVERED' if f1_s4 >= f1_s1 * 0.95 else ('↑ IMPROVED' if recovery > 0.05 else '— PARTIAL')
    
    per_class_recovery.append({
        'class': cls_name, 'f1_s1': f1_s1, 'f1_s2': f1_s2, 
        'f1_s3': f1_s3, 'f1_s4': f1_s4, 'recovery': recovery, 'status': status
    })
    
    print(f'{cls_name:<25s} {f1_s1*100:>7.2f}% {f1_s2*100:>7.2f}% {f1_s3*100:>7.2f}% {f1_s4*100:>7.2f}% '
          f'{recovery*100:>9.2f}% {status:>10s}')

print('='*90)
recovered = [r for r in per_class_recovery if '✓' in r['status']]
print(f'\nFully recovered classes: {len(recovered)}/{len(per_class_recovery)}')

## 7. Multi-Epsilon Adversarial Training Comparison

Bandingkan model yang di-train dengan ε=0.01 vs ε=0.1 untuk melihat
apakah ada perbedaan robustness.

In [ ]:
# Train model robust dengan ε=0.01 (conservative)
print('Training conservative model (ε=0.01)...')

# Generate adversarial training data dengan ε=0.01
X_adv_conservative = adv_data.get('X_adv_eps01_train', None)

if X_adv_conservative is not None:
    n_adv_cons = min(n_adv_target, len(X_adv_conservative))
    X_robust_cons = np.vstack([X_train_clean, X_adv_conservative[:n_adv_cons]])
    y_robust_cons = np.concatenate([y_train_clean, y_adv_train[:n_adv_cons]])
    
    # Shuffle
    shuffle_cons = np.random.RandomState(RANDOM_SEED).permutation(len(X_robust_cons))
    X_robust_cons = X_robust_cons[shuffle_cons]
    y_robust_cons = y_robust_cons[shuffle_cons]
    
    model_robust_cons = XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', random_state=RANDOM_SEED,
        n_jobs=-1, tree_method='hist'
    )
    model_robust_cons.fit(X_robust_cons, y_robust_cons)
    
    # Evaluate
    r_cons_clean = full_evaluation(model_robust_cons, X_test_clean, y_test, 'Robust(ε=0.01)+Clean')
    r_cons_adv = full_evaluation(model_robust_cons, X_test_adv, y_test, 'Robust(ε=0.01)+Adv')
    
    print(f'  Robust(ε=0.01) + Clean:  MCC={r_cons_clean["mcc"]:.4f} | F1={r_cons_clean["f1_score"]*100:.2f}%')
    print(f'  Robust(ε=0.01) + Adv:    MCC={r_cons_adv["mcc"]:.4f} | F1={r_cons_adv["f1_score"]*100:.2f}%')
    print(f'  vs Robust(ε=0.10) + Adv: MCC={results_2x2["S4"]["mcc"]:.4f} | F1={results_2x2["S4"]["f1_score"]*100:.2f}%')
else:
    print('  Conservative adversarial data not available, skipping comparison.')
    r_cons_clean = None
    r_cons_adv = None

## 8. Simpan Hasil

In [ ]:
# Save comprehensive results
robust_output = {
    'results_2x2': {k: {key: val for key, val in v.items() if key not in ['y_pred', 'confusion_matrix']}
                    for k, v in results_2x2.items()},
    'confusion_matrices': {k: v['confusion_matrix'].tolist() for k, v in results_2x2.items()},
    'per_class_recovery': per_class_recovery,
    'training_config': {
        'epsilon': EPSILON,
        'augmentation_ratio': f'{n_clean}:{n_adv_use} (clean:adv)',
        'n_estimators': 200,
        'max_depth': 8,
        'learning_rate': 0.1,
        'train_time_sec': train_time,
        'model_size_mb': model_size
    },
    'summary': {
        'security_gap_mcc': security_gap,
        'recovery_mcc': recovery,
        'integrity_loss_mcc': integrity_loss
    },
    'conservative_comparison': {
        'eps001_clean_mcc': r_cons_clean['mcc'] if r_cons_clean else None,
        'eps001_adv_mcc': r_cons_adv['mcc'] if r_cons_adv else None,
        'eps010_clean_mcc': results_2x2['S3']['mcc'],
        'eps010_adv_mcc': results_2x2['S4']['mcc']
    }
}

with open(os.path.join(DATA_DIR, 'robust_results_06.pkl'), 'wb') as f:
    pickle.dump(robust_output, f)

print('Saved:')
print(f'  {MODEL_DIR}robust_xgboost_top10.json (model)')
print(f'  {DATA_DIR}robust_results_06.pkl (results)')
print(f'  {DATA_DIR}adversarial_training_2x2_comparison.png')
print(f'  {DATA_DIR}confusion_s2_vs_s4.png')

## 9. Narasi & Kesimpulan

In [ ]:
print('='*70)
print(f'{"NARASI: ADVERSARIAL TRAINING EFFECTIVENESS":^70}')
print('='*70)
print(f'''
■ HASIL UTAMA:

  1. SECURITY GAP (S1 → S2):
     MCC turun dari {results_2x2['S1']['mcc']:.4f} → {results_2x2['S2']['mcc']:.4f}
     Gap = {security_gap:.4f} — menunjukkan kerentanan serius
     
  2. INTEGRITY CHECK (S1 vs S3):
     MCC Baseline clean: {results_2x2['S1']['mcc']:.4f}
     MCC Robust clean:   {results_2x2['S3']['mcc']:.4f}
     Loss = {integrity_loss:.4f} — {'✓ MINIMAL (model tetap akurat pada traffic normal)' if abs(integrity_loss) < 0.02 else '⚠ Ada sedikit trade-off'}
     
  3. ROBUSTNESS RECOVERY (S2 → S4):
     MCC Baseline+Adv: {results_2x2['S2']['mcc']:.4f}
     MCC Robust+Adv:   {results_2x2['S4']['mcc']:.4f}
     Recovery = +{recovery:.4f} — {'✓ SIGNIFIKAN' if recovery > 0.1 else '↑ Moderate improvement'}

■ KESIMPULAN:
  → Adversarial Training BERHASIL memperkuat model
  → Model robust mempertahankan integritas pada traffic normal (S3 ≈ S1)
  → Sekaligus meningkatkan ketahanan terhadap evasion (S4 >> S2)
  → Trade-off antara efisiensi fitur dan keamanan BISA dimitigasi
    melalui augmentasi data adversarial

■ NEXT STEP:
  → Notebook 07: Robustness Ablation Study (Top-5/10/15/Full × Robust)
  → Notebook 08: Evaluasi detail + visualisasi final untuk paper

{'='*70}
''')